# Session 12 — Scalable End-to-End MLOps Pipelines using Google Vertex AI with Smart Analytics

**Goal:** move beyond a single AutoML training call (Session 4) to a full
**Vertex AI Pipeline** — a directed graph of steps (preprocess → train → evaluate →
conditionally deploy) that runs as one orchestrated, repeatable job, plus a BigQuery
analytics layer for monitoring predictions at scale.

## Why a pipeline instead of a notebook you re-run by hand

Sessions 1-4 each did one thing well, but a real production system chains many steps
with dependencies: don't train on data that failed validation (Session 11), don't
deploy a model that scores worse than what's currently live. A **Vertex AI Pipeline**
(built on Kubeflow Pipelines) encodes that dependency graph explicitly and re-runs
reproducibly, with each step's inputs/outputs tracked automatically.

## Prerequisites

Needs a **GCP project** with Vertex AI, Cloud Build, and BigQuery enabled — not
available in this sandbox. Complete, correct reference code below.

```bash
pip install google-cloud-aiplatform kfp
```

In [ ]:
PROJECT_ID = "your-gcp-project-id"
REGION = "us-central1"
PIPELINE_ROOT = "gs://your-bucket/pipeline-root"

## Step 1 — Define pipeline components

Each `@component` is a containerized, independently-runnable function — Vertex AI
packages it, tracks its inputs/outputs as pipeline artifacts, and can retry or cache
it independently of the other steps.

In [ ]:
from kfp import dsl
from kfp.dsl import component, Input, Output, Dataset, Model, Metrics

@component(base_image="python:3.11", packages_to_install=["pandas", "scikit-learn"])
def preprocess(raw_data_path: str, output_dataset: Output[Dataset]):
    import pandas as pd
    df = pd.read_csv(raw_data_path)
    df = df.dropna()
    df.to_csv(output_dataset.path, index=False)


@component(base_image="python:3.11", packages_to_install=["pandas", "scikit-learn", "joblib"])
def train(dataset: Input[Dataset], model: Output[Model], metrics: Output[Metrics]):
    import pandas as pd, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score

    df = pd.read_csv(dataset.path)
    X, y = df.drop(columns="target"), df["target"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    clf = RandomForestClassifier(n_estimators=200, random_state=0).fit(X_train, y_train)
    auc = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])

    metrics.log_metric("test_auc", auc)
    joblib.dump(clf, model.path)


@component(base_image="python:3.11")
def evaluate_gate(metrics: Input[Metrics], min_auc: float) -> bool:
    auc = metrics.metadata["test_auc"]
    print(f"Trained model AUC: {auc}, minimum required: {min_auc}")
    return auc >= min_auc

## Step 2 — Assemble the components into a pipeline graph

`@dsl.pipeline` wires the components together; the `dsl.If` block means the deploy
step only runs if `evaluate_gate` returns `True` — a newly trained model that
underperforms never reaches production.

In [ ]:
@dsl.pipeline(name="heart-disease-e2e-pipeline", pipeline_root=PIPELINE_ROOT)
def heart_disease_pipeline(raw_data_path: str, min_auc: float = 0.85):
    preprocess_task = preprocess(raw_data_path=raw_data_path)
    train_task = train(dataset=preprocess_task.outputs["output_dataset"])
    gate_task = evaluate_gate(metrics=train_task.outputs["metrics"], min_auc=min_auc)

    with dsl.If(gate_task.output == True):
        from kfp.dsl import importer
        deploy_task = deploy_model(model=train_task.outputs["model"])

## Step 3 — Compile and submit the pipeline

Compiling produces a JSON/YAML pipeline definition; submitting runs it on Vertex AI's
managed pipeline service, which schedules each component as its own container run.

In [ ]:
from kfp import compiler
from google.cloud import aiplatform

compiler.Compiler().compile(heart_disease_pipeline, "heart_disease_pipeline.json")

aiplatform.init(project=PROJECT_ID, location=REGION)
job = aiplatform.PipelineJob(
    display_name="heart-disease-e2e-run",
    template_path="heart_disease_pipeline.json",
    parameter_values={
        "raw_data_path": "gs://your-bucket/heart_disease_raw.csv",
        "min_auc": 0.85,
    },
)
job.submit()
print(f"Pipeline submitted: {job.resource_name}")

## Step 4 — Smart analytics: stream predictions into BigQuery

"Smart analytics" here means treating every prediction as a row in a queryable
warehouse table, so trends (prediction volume, class balance over time, latency) are
a SQL query away instead of scattered across log files.

In [ ]:
from google.cloud import bigquery

bq_client = bigquery.Client(project=PROJECT_ID)

schema = [
    bigquery.SchemaField("prediction_time", "TIMESTAMP"),
    bigquery.SchemaField("model_version", "STRING"),
    bigquery.SchemaField("predicted_class", "INTEGER"),
    bigquery.SchemaField("predicted_probability", "FLOAT"),
]
table_id = f"{PROJECT_ID}.mlops_monitoring.predictions_log"
table = bigquery.Table(table_id, schema=schema)
bq_client.create_table(table, exists_ok=True)

query = f'''\
SELECT DATE(prediction_time) AS day,
       model_version,
       COUNT(*) AS n_predictions,
       AVG(predicted_probability) AS avg_confidence
FROM `{table_id}`
GROUP BY day, model_version
ORDER BY day DESC
'''
print("Once predictions are logged, this query gives a daily monitoring dashboard:")
print(query)

## Step 5 — Real-time model monitoring

Vertex AI's built-in **Model Monitoring** attaches directly to a deployed endpoint,
comparing live traffic against training data automatically — a managed alternative to
running Evidently (Session 5) yourself on a schedule.

In [ ]:
from google.cloud.aiplatform import model_monitoring

monitoring_job = aiplatform.ModelDeploymentMonitoringJob.create(
    display_name="heart-disease-monitoring",
    endpoint="projects/.../locations/.../endpoints/YOUR_ENDPOINT_ID",
    logging_sampling_strategy=model_monitoring.RandomSampleConfig(sample_rate=0.5),
    schedule_config=model_monitoring.ScheduleConfig(monitor_interval=24),
    objective_configs=model_monitoring.ObjectiveConfig(
        skew_detection_config=model_monitoring.SkewDetectionConfig(
            data_source="gs://your-bucket/heart_disease_raw.csv",
            target_field="target",
        ),
    ),
)
print("Monitoring job created:", monitoring_job.resource_name)

## What to try next

* Add a Slack/email alert component that fires when `evaluate_gate` returns `False`,
  so a failed quality gate notifies the team instead of silently skipping deployment.
* Session 13 extends this same pipeline shape to cover LLM-based applications
  specifically. Session 14 focuses on the retraining trigger this pipeline is missing
  (currently it only runs when someone submits it manually).